# Clase 5 — Prompting avanzado, salidas JSON y decisiones estructuradas

> **¿Cómo pasamos de una instrucción ambigua a una salida que otra aplicación pueda usar?**

Esta clase trabaja con OpenRouter. Primero comparamos prompts simples y robustos, usamos ejemplos few-shot y construimos respuestas JSON para automatizar tareas administrativas. Después introducimos un tipo distinto de modelo: **Jev**, un modelo *System One* que en lugar de generar texto devuelve decisiones tipadas con probabilidades. Cerramos comparando ambos enfoques para clasificar los mismos tickets.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Descomponer un prompt en rol, tarea, contexto, restricciones y formato.
- Explicar la diferencia entre zero-shot y few-shot prompting.
- Pedir una justificación breve y verificable sin solicitar el razonamiento interno del modelo.
- Diseñar y validar una salida JSON para una automatización administrativa.
- Explicar qué es un modelo System One y en qué se diferencia de un LLM conversacional.
- Comparar un LLM con system prompt frente a un modelo de decisión tipada (Jev) para clasificar tickets.
- Identificar cuándo el resultado debe pasar a revisión humana.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Preparar OpenRouter y probar una consulta |
| 2 | Anatomía de un prompt robusto |
| 3 | Zero-shot frente a few-shot |
| 4 | Razonamiento visible, breve y verificable |
| 5 | JSON y validación local |
| 6 | Modelos System One y Jev: decisiones tipadas |
| 7 | Laboratorio: LLM con system prompt frente a Jev |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en orden.
2. Observá primero el resultado antes de cambiar las variables `TODO`.
3. Ingresá tu clave de OpenRouter en la celda indicada; se guarda en el entorno de esta sesión.
4. Cada consulta de este notebook usa OpenRouter y requiere conexión a Internet.
5. Usá solamente textos ficticios. No pegues historias clínicas ni datos personales reales.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Prompt | Instrucción y contexto que recibe el modelo |
| Zero-shot | Resolver una tarea sin ejemplos en la instrucción |
| Few-shot | Resolver una tarea mostrando ejemplos de entrada y salida |
| Restricción | Regla que limita el comportamiento esperado |
| Salida estructurada | Resultado con campos y tipos previsibles |
| JSON | Formato de texto para intercambiar datos estructurados |
| Esquema | Descripción de campos, tipos y valores permitidos |
| Validación | Comprobación de sintaxis y condiciones mínimas |
| Revisión humana | Control de una persona antes de tomar una decisión |
| Prompt injection | Texto que intenta modificar las instrucciones de la aplicación |
| LLM | Modelo de lenguaje que genera texto libre |
| System One | Modelo que devuelve decisiones tipadas con probabilidades, sin texto libre |
| Choice | Pregunta que elige una opción de un conjunto definido |
| Noul | Pregunta de sí/no que devuelve la probabilidad de sí |
| Score | Pregunta que ubica la entrada en una escala ordenada |
| Confianza | Grado de concentración de la distribución de probabilidades |

## Antes de empezar: cuenta, clave y costos

OpenRouter funciona como una puerta de entrada a distintos modelos. Para esta clase necesitamos una cuenta y una clave de API. La clave se guarda como variable de entorno.

In [2]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Clave de OpenRouter: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
print("Clave configurada:", bool(OPENROUTER_API_KEY))

Clave configurada: True


In [44]:
import json
import os
import ssl
from urllib.request import Request, urlopen

import certifi

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-6-luna")
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())

# Configuración para Jev (modelo System One) a través de la Decisions API
OPENROUTER_DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"
MODEL_JEV = "typesafe/jev-1.13"

## 1. Una función simple para consultar el modelo

OpenRouter recibe el `system prompt` dentro de `messages`, con `"role": "system"`. El contexto SSL verifica la conexión HTTPS usando los certificados de `certifi`.


In [28]:
def consultar_modelo(messages, temperature=0.2, max_tokens=250, response_format=None):
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "reasoning" : {"effort": "none"}
    }
    if response_format is not None:
        payload["response_format"] = response_format

    solicitud = Request(
        OPENROUTER_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
    )
    with urlopen(solicitud, context=SSL_CONTEXT, timeout=60) as respuesta:
        return json.load(respuesta)["choices"][0]["message"]["content"].strip()


## 2. Anatomía de un prompt robusto

Un prompt útil se parece más a un contrato que a una pregunta suelta:

```text
rol + tarea + contexto + restricciones + criterio de desconocido + formato
```

La calidad no depende de hacer el prompt cada vez más largo. Depende de quitar ambigüedad en los puntos que afectan la decisión.

In [7]:
MENSAJE_ADMINISTRATIVO = "Necesito un turno con dermatología el martes por la tarde."

prompt_simple = [
    {"role": "system", "content": ("Sos un asistente de recepción. Respondé de forma breve y clara")},
    {"role": "user", "content": MENSAJE_ADMINISTRATIVO}
]

prompt_robusto = [
    {"role": "system", "content": (
        "Sos un clasificador de solicitudes de recepción. "
        "Elegí una sola intención: turno, horarios, receta, consulta_medica u otro. "
        "No inventes datos. Si falta información, indicá cuál. "
        "No diagnostiques ni indiques tratamientos."
    )},
    {"role": "user", "content": (
        f"Solicitud: {MENSAJE_ADMINISTRATIVO}\n"
        "Respondé con la intención y los datos que habría que registrar."
    )},
]

print("Prompt simple:")
print(consultar_modelo(prompt_simple))
print("-" * 60)
print("\nPrompt robusto:")
print(consultar_modelo(prompt_robusto))

Prompt simple:
Claro. ¿Te referís al próximo martes? ¿Preferís algún horario en particular por la tarde?
------------------------------------------------------------

Prompt robusto:
**Intención:** turno  
**Datos para registrar:** especialidad: dermatología; preferencia: martes por la tarde.  
**Falta confirmar:** la fecha exacta del martes y los datos de contacto del paciente.


In [8]:
print(f"Modelo: {MODEL}")
print(f"Proveedor: {OPENROUTER_URL}")

Modelo: openai/gpt-6-luna
Proveedor: https://openrouter.ai/api/v1/chat/completions


### Para interpretar

- ¿Qué información agregamos en el prompt robusto?
- ¿Qué regla evita que el sistema invente horarios?
- ¿Qué parte del prompt cambiarías si el sistema fuera para una biblioteca?

Un buen prompt reduce decisiones ambiguas, pero no reemplaza las validaciones de la aplicación.

## 3. Zero-shot frente a few-shot

En **zero-shot** describimos la tarea, pero no mostramos ejemplos. En **few-shot** agregamos ejemplos de entrada y salida para enseñar el criterio esperado.

Los ejemplos funcionan como una pequeña tabla de casos. No son entrenamiento permanente: solo orientan esta consulta. Deben ser representativos, consistentes y no contradictorios.

In [19]:
def construir_prompt_clasificacion(mensaje, incluir_ejemplos=False):
    mensajes = [{"role": "system", "content": (
        "Clasificá solicitudes de recepción. Categorías permitidas: turno, horarios, receta, consulta_medica u otro. "
        "Devolvé solo la categoría elegida y no inventes información."
    )}]
    if incluir_ejemplos:
        mensajes.extend([
            {"role": "user", "content": "Quiero cambiar mi turno para el viernes."},
            {"role": "assistant", "content": "turno"},
            {"role": "user", "content": "¿A qué hora abre el consultorio?"},
            {"role": "assistant", "content": "horarios"},
        ])
    mensajes.append({"role": "user", "content": mensaje})
    return mensajes

casos = [
    "Quiero solicitar un turno para la semana próxima.",
    "¿Atienden el sábados?",
    "Tengo dolor intenso y necesito saber qué medicamento tomar.",
    "Atiende el Dr. Medina?",
    "Necesito renovar una receta y pedir turno, tengo un dolor fuerte.",
    "Reciben mercado pago",
]

for caso in casos:
    zero_shot = consultar_modelo(construir_prompt_clasificacion(caso, incluir_ejemplos=False))
    few_shot = consultar_modelo(construir_prompt_clasificacion(caso, incluir_ejemplos=True))
    print("Caso:", caso)
    print("  zero-shot:", zero_shot)
    print("  few-shot:", few_shot)

Caso: Quiero solicitar un turno para la semana próxima.
  zero-shot: turno
  few-shot: turno
Caso: ¿Atienden el sábados?
  zero-shot: horarios
  few-shot: horarios
Caso: Tengo dolor intenso y necesito saber qué medicamento tomar.
  zero-shot: consulta_medica
  few-shot: consulta_medica
Caso: Atiende el Dr. Medina?
  zero-shot: horarios
  few-shot: consulta_medica
Caso: Necesito renovar una receta y pedir turno, tengo un dolor fuerte.
  zero-shot: consulta_medica
  few-shot: consulta_medica
Caso: Reciben mercado pago
  zero-shot: otro
  few-shot: otro


### Actividad breve

Agregá dos ejemplos al prompt few-shot. Uno debe ser claramente `receta` y otro debe ser `otro`. Luego probá un mensaje nuevo.

Preguntate:

- ¿El ejemplo enseña realmente el criterio o solo repite palabras?
- ¿Qué ocurre si dos ejemplos parecen pertenecer a categorías distintas?
- ¿Cuántos ejemplos hacen falta antes de que el prompt se vuelva costoso y difícil de mantener?

## 4. Razonamiento: pedir evidencia breve, no una cadena privada

A veces queremos saber por qué el modelo eligió una categoría. Para una aplicación conviene pedir una **justificación breve basada en criterios observables**, por ejemplo: `menciona una solicitud de turno y una fecha`.

No hace falta pedir que el modelo revele todos sus pasos internos. Una cadena de pensamiento extensa puede incluir información irrelevante, ser difícil de verificar y no garantiza que la respuesta sea correcta. La aplicación debe validar la salida y, cuando corresponda, escalarla a una persona.

In [36]:
prompt_con_evidencia = [
    {"role": "system", "content": (
        "Clasificá la solicitud. Respondé con una categoría y una justificación de una sola oración. "
        "La justificación debe mencionar únicamente señales observables del mensaje; no describas razonamientos internos."
    )},
    {"role": "user", "content": "Ya volvio el Dr. Juarez?"}
]

print(consultar_modelo(prompt_con_evidencia, temperature=0.1, max_tokens=100))
print(MODEL)

Categoría: Consulta de estado. Justificación: El mensaje pregunta si el Dr. Juárez ya volvió.
openai/gpt-6-luna


## 5. De texto libre a JSON

Un texto puede ser útil para una persona, pero difícil de conectar con un sistema. JSON permite separar campos y automatizar pasos posteriores.

Nuestro contrato mínimo será:

```json
ESQUEMA_SALIDA:

{
  "intencion": "turno",
  "datos_extraidos": {},
  "datos_faltantes": [],
  "respuesta_sugerida": "...",
  "justificacion_breve": "...",
  "requiere_revision_humana": false
}
```

El JSON válido es solo el primer control. También hay que comprobar claves, tipos, valores permitidos y reglas del negocio.

In [43]:
def construir_prompt_json(mensaje):
    return [
        {"role": "system", "content": (
            "Respondé únicamente con un objeto JSON válido que respete EXACTAMENTE este esquema:\n"
            + json.dumps(ESQUEMA_SALIDA, ensure_ascii=False)
        )},
        {"role": "user", "content": f"Mensaje ficticio: {mensaje}"}
    ]
print(construir_prompt_json(mensaje))

[{'role': 'system', 'content': 'Respondé únicamente con un objeto JSON válido que respete EXACTAMENTE este esquema:\n{"type": "object", "properties": {"intencion": {"type": "string", "enum": ["turno", "horarios", "receta", "consulta_medica", "otro"]}, "datos_extraidos": {"type": "object"}, "datos_faltantes": {"type": "array", "items": {"type": "string"}}, "respuesta_sugerida": {"type": "string"}, "justificacion_breve": {"type": "string"}, "requiere_revision_humana": {"type": "boolean"}}, "required": ["intencion", "datos_extraidos", "datos_faltantes", "respuesta_sugerida", "justificacion_breve", "requiere_revision_humana"], "additionalProperties": false}'}, {'role': 'user', 'content': 'Mensaje ficticio: Quería saber si tienen estacionamiento.'}]


### Dos niveles de validación

1. **Sintáctica:** la respuesta se puede leer con `json.loads`.
2. **Semántica:** los campos existen, tienen tipos válidos y respetan reglas mínimas.

Todavía falta una tercera capa: verificar si la información es verdadera y suficiente para el proceso. Esa parte puede requerir una base de datos, reglas del negocio o una persona.

## 6. Modelos System One y Jev: decisiones tipadas

Hasta acá le pedimos a un LLM que **escriba** una respuesta (texto o JSON) y después la validamos. Ese flujo se llama *prompt-and-parse*: el modelo genera, nosotros parseamos. Funciona, pero tiene un costo: el modelo puede desviarse del formato, inventar campos o dar respuestas ambiguas.

Existe otra familia de modelos pensada para el caso opuesto: en lugar de generar texto, **deciden**. Se llaman **System One**, por la analogía con el "pensamiento rápido" de Kahneman. Un modelo System One recibe un estado y responde preguntas tipadas con probabilidades. No escribe prosa ni explica su razonamiento: devuelve una decisión que tu código puede usar directamente.

### Las tres primitivas

| Primitiva | Pregunta que responde | Qué devuelve |
|---|---|---|
| **Choice** | ¿Cuál de estas opciones? | La opción elegida, una probabilidad por opción y una confianza |
| **Noul** | ¿Se cumple esta condición? | La probabilidad de que sea sí |
| **Score** | ¿Dónde cae en una escala ordenada? | Una posición ponderada, probabilidad por nivel y confianza |

### Jev en OpenRouter

**Jev** (`typesafe/jev-1.13`) es el primer modelo System One de TypeSafe y está disponible por OpenRouter. No se llama por el endpoint de chat completions, sino por la **Decisions API**, con un contrato distinto:

```text
POST https://openrouter.ai/api/alpha/decisions
{
  "model": "typesafe/jev-1.13",
  "state": { ... },        # el contenido a evaluar
  "questions": { ... }     # preguntas tipadas (choice, noul, score)
}
```

La respuesta trae una entrada por pregunta, con probabilidades y confianza, más el costo de la llamada:

```json
{
  "answers": {
    "intencion": { "type": "choice", "choice": "turno", "probabilities": {...}, "confidence": 1 },
    "requiere_revision": { "type": "noul", "noul": 0.21 }
  },
  "usage": { "input_tokens": 457, "output_tokens": 79, "cost": 0.000019194 }
}
```

### Por qué importa la incertidumbre

Un LLM te da una etiqueta sin decirte qué tan seguro está. Jev te da una **probabilidad**: si `noul` vale `0.96`, está casi seguro de que sí; si vale `0.51`, es un tiro al aire. Esa señal es oro para decidir cuándo **escalar a una persona** en vez de actuar automáticamente. La confianza describe la distribución de las alternativas, no si el flujo es seguro: los umbrales se eligen según el costo de cada tipo de error.

> **Nota:** Jev no reemplaza a un LLM. Si necesitás prosa, explicaciones o redacción, usá un modelo conversacional. Si necesitás una decisión tipada y predecible, Jev es una alternativa directa al paso "prompt + parse".

In [45]:
def consultar_jev(state, questions, timeout=60):
    """Consulta a Jev (System One) por la Decisions API de OpenRouter.

    Devuelve el diccionario completo de la respuesta (answers + usage).
    Lanza RuntimeError si la petición falla o la respuesta no trae 'answers'.
    """
    payload = {
        "model": MODEL_JEV,
        "state": state,
        "questions": questions,
    }
    solicitud = Request(
        OPENROUTER_DECISIONS_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
    )
    with urlopen(solicitud, context=SSL_CONTEXT, timeout=timeout) as respuesta:
        datos = json.load(respuesta)
    if "answers" not in datos:
        raise RuntimeError(f"Respuesta inesperada de Jev: {datos}")
    return datos

In [46]:
# Definimos las preguntas tipadas que Jev responderá sobre cada ticket.
# Choice elige la intención; Noul decide si hace falta revisión humana.
PREGUNTAS_JEV = {
    "intencion": {
        "type": "choice",
        "instructions": "¿Cuál es la intención de esta solicitud de recepción?",
        "criteria": {
            "turno": "Solicitud de turno o cambio de turno.",
            "horarios": "Pregunta por horarios de atención.",
            "receta": "Solicitud de receta.",
            "consulta_medica": "Consulta clínica o urgencia.",
            "otro": "Cualquier otra cosa.",
        },
    },
    "requiere_revision": {
        "type": "noul",
        "instructions": "¿Esta solicitud requiere revisión humana?",
        "criteria": {
            "true": "Es una consulta clínica, urgencia o caso ambiguo.",
            "false": "Es una solicitud administrativa simple.",
        },
    },
}

ticket_ejemplo = "Necesito un turno con dermatología el martes por la tarde."
respuesta_je = consultar_jev({"ticket": ticket_ejemplo}, PREGUNTAS_JEV)
print(json.dumps(respuesta_je, ensure_ascii=False, indent=2))

{
  "model": "typesafe/jev-1.13-20260917",
  "answers": {
    "intencion": {
      "type": "choice",
      "choice": "turno",
      "probabilities": {
        "receta": 0,
        "otro": 0,
        "consulta_medica": 0,
        "turno": 1,
        "horarios": 0
      },
      "confidence": 1
    },
    "requiere_revision": {
      "type": "noul",
      "noul": 0.2
    }
  },
  "usage": {
    "input_tokens": 457,
    "output_tokens": 79,
    "cost": 1.9194e-05
  },
  "id": "gen-dec-1790612440-YYyoAha8RJoG5cN3ThpX",
  "provider": "TypeSafe"
}


### Para interpretar

- `intencion.choice` es la opción elegida; `intencion.probabilities` reparte la probabilidad entre las alternativas y `intencion.confidence` resume qué tan concentrada está esa distribución.
- `requiere_revision.noul` es la **probabilidad de que la respuesta sea sí**. Un valor bajo (p. ej. `0.21`) indica que Jev cree que no hace falta revisión humana.
- ¿Qué diferencia ves con la salida JSON del LLM de la sección anterior? ¿Cuál te da más información para decidir si escalar a una persona?

## 7. Laboratorio: LLM con system prompt frente a Jev

Procesamos la misma colección de mensajes ficticios de recepción con **dos sistemas distintos**:

1. **LLM con system prompt** (el baseline de la sección 5): genera un JSON que después validamos.
2. **Jev** (System One): responde preguntas tipadas con probabilidades.

El objetivo no es declarar un ganador con una muestra tan chica. Es observar **qué tipo de error comete cada uno**, cuál es más fácil de validar y cómo la incertidumbre de Jev puede guiar la revisión humana.

### Tu tarea

- Cambiá o agregá mensajes en `CASOS_LABORATORIO`.
- Revisá si la intención coincide con el texto.
- Compará la etiqueta de cada modelo contra la referencia.
- Fijate en qué casos Jev da baja confianza o alta probabilidad de revisión.
- Explicá por qué algunos casos no deberían resolverse automáticamente.

In [34]:
# Cada caso lleva una etiqueta de referencia (la intención esperada).
# Agregá o cambiá casos y su etiqueta para explorar.
CASOS_LABORATORIO = [
    ("Quiero un turno con cardiología para el viernes a la mañana.", "turno"),
    ("¿Cuál es el horario de atención del consultorio?", "horarios"),
    ("¿Pueden renovar mi receta de forma urgente?", "receta"),
    ("Tengo dolor en el pecho desde hace una hora, ¿qué tomo?", "consulta_medica"),
    ("Quería saber si tienen estacionamiento.", "otro"),
]

resultados = []
for mensaje, referencia in CASOS_LABORATORIO:
    fila = {"mensaje": mensaje, "referencia": referencia}

    # --- Sistema 1: LLM con system prompt + JSON ---
    try:
        salida = consultar_modelo(
            construir_prompt_json(mensaje),
            temperature=0.1,
            max_tokens=300,
            response_format={"type": "json_object"},
        )
        registro = json.loads(salida)
        valido, detalle = validar_registro(registro)
        fila["llm_intencion"] = registro.get("intencion")
        fila["llm_revision"] = registro.get("requiere_revision_humana")
        fila["llm_valido"] = valido
        fila["llm_detalle"] = detalle
    except (json.JSONDecodeError, RuntimeError) as problema:
        fila["llm_intencion"] = None
        fila["llm_revision"] = None
        fila["llm_valido"] = False
        fila["llm_detalle"] = str(problema)

    # --- Sistema 2: Jev (System One) con preguntas tipadas ---
    try:
        respuesta_je = consultar_jev({"ticket": mensaje}, PREGUNTAS_JE)
        answers = respuesta_je["answers"]
        fila["jev_intencion"] = answers["intencion"]["choice"]
        fila["jev_confianza"] = answers["intencion"].get("confidence")
        fila["jev_probabilidades"] = answers["intencion"].get("probabilities")
        fila["jev_revision"] = answers["requiere_revision"]["noul"]
        fila["jev_costo"] = respuesta_je.get("usage", {}).get("cost")
    except (KeyError, RuntimeError) as problema:
        fila["jev_intencion"] = None
        fila["jev_confianza"] = None
        fila["jev_probabilidades"] = None
        fila["jev_revision"] = None
        fila["jev_costo"] = None
        fila["jev_error"] = str(problema)

    resultados.append(fila)

for fila in resultados:
    print(json.dumps(fila, ensure_ascii=False, indent=2))
    print("-" * 60)

{
  "mensaje": "Quiero un turno con cardiología para el viernes a la mañana.",
  "referencia": "turno",
  "llm_intencion": "turno",
  "llm_revision": false,
  "llm_valido": true,
  "llm_detalle": "Registro válido para la siguiente etapa",
  "jev_intencion": "turno",
  "jev_confianza": 1,
  "jev_probabilidades": {
    "receta": 0,
    "horarios": 0,
    "consulta_medica": 0,
    "otro": 0,
    "turno": 1
  },
  "jev_revision": 0.18,
  "jev_costo": 1.9152e-05
}
------------------------------------------------------------
{
  "mensaje": "¿Cuál es el horario de atención del consultorio?",
  "referencia": "horarios",
  "llm_intencion": "horarios",
  "llm_revision": false,
  "llm_valido": true,
  "llm_detalle": "Registro válido para la siguiente etapa",
  "jev_intencion": "horarios",
  "jev_confianza": 1,
  "jev_probabilidades": {
    "otro": 0,
    "turno": 0,
    "consulta_medica": 0,
    "receta": 0,
    "horarios": 1
  },
  "jev_revision": 0.07,
  "jev_costo": 1.9068e-05
}
--------------

In [35]:
# Resumen comparativo: exactitud, validez de formato y desacuerdos.
def exactitud(fila, clave):
    return fila.get(clave) == fila["referencia"]

aciertos_llm = sum(exactitud(f, "llm_intencion") for f in resultados)
aciertos_jev = sum(exactitud(f, "jev_intencion") for f in resultados)
total = len(resultados)

print(f"Total de casos: {total}")
print(f"LLM  -> aciertos: {aciertos_llm}/{total}")
print(f"Jev  -> aciertos: {aciertos_jev}/{total}")
print()

print("Desacuerdos entre modelos:")
for f in resultados:
    if f.get("llm_intencion") != f.get("jev_intencion"):
        print(f"  - {f['mensaje']}")
        print(f"      LLM: {f.get('llm_intencion')} | Jev: {f.get('jev_intencion')} | Ref: {f['referencia']}")
print()

print("Casos que Jev escalaría a revisión (noul alto):")
for f in resultados:
    noul = f.get("jev_revision")
    if noul is not None and noul >= 0.5:
        print(f"  - {f['mensaje']} (noul={noul:.2f})")

Total de casos: 5
LLM  -> aciertos: 4/5
Jev  -> aciertos: 5/5

Desacuerdos entre modelos:
  - ¿Pueden renovar mi receta de forma urgente?
      LLM: consulta_medica | Jev: receta | Ref: receta

Casos que Jev escalaría a revisión (noul alto):
  - ¿Pueden renovar mi receta de forma urgente? (noul=0.87)
  - Tengo dolor en el pecho desde hace una hora, ¿qué tomo? (noul=0.98)


## 8. Análisis de la comparación

Los dos sistemas resuelven la misma tarea con contratos muy distintos. Antes de elegir uno, conviene pensar en qué se gana y qué se pierde con cada enfoque.

### Preguntas de análisis

- ¿En qué casos coincidieron LLM y Jev? ¿En cuáles se desacordaron?
- ¿Qué clase fue más difícil para cada modelo?
- ¿Qué información extra da Jev que el LLM no da? ¿Cómo usarías la confianza o el `noul` para decidir cuándo escalar a una persona?
- ¿Qué pasa si el LLM devuelve JSON inválido? ¿Y si Jev devuelve una categoría fuera del conjunto? ¿Cuál es más fácil de validar?
- ¿Cuándo preferirías un LLM con system prompt y cuándo un modelo System One?

### Límites de esta comparación

- La muestra es didáctica y chica: no alcanza para declarar un ganador estadístico.
- Los contratos son distintos: el LLM genera texto que hay que parsear; Jev devuelve una decisión tipada. Comparar solo la etiqueta elegida deja afuera esa diferencia.
- Jev no explica su razonamiento. Si necesitás una justificación escrita, podés usar Jev para decidir y un LLM para redactar el motivo, o escalar los casos de baja confianza a una persona.

In [32]:
# Caso clínico: Jev debería marcarlo como consulta_medica y con noul alto.
mensaje_salud = "Tengo fiebre y dolor fuerte desde anoche, ¿qué medicamento debería tomar?"
respuesta_salud = consultar_jev({"ticket": mensaje_salud}, PREGUNTAS_JE)
answers = respuesta_salud["answers"]

print("Intención:", answers["intencion"]["choice"])
print("Confianza:", answers["intencion"].get("confidence"))
print("Probabilidades:", answers["intencion"].get("probabilities"))
print("Probabilidad de revisión humana (noul):", answers["requiere_revision"]["noul"])
print("Costo (USD):", respuesta_salud.get("usage", {}).get("cost"))

Intención: consulta_medica
Confianza: 0.99
Probabilidades: {'receta': 0.01, 'horarios': 0, 'otro': 0, 'consulta_medica': 0.99, 'turno': 0}
Probabilidad de revisión humana (noul): 0.96
Costo (USD): 1.9236e-05


## Checklist para depurar un prompt

Antes de confiar en una automatización, revisá:

- ¿La tarea está escrita en una sola frase clara?
- ¿Las categorías y valores permitidos están definidos?
- ¿Hay ejemplos representativos y no contradictorios?
- ¿Está claro qué hacer cuando falta información?
- ¿El formato se valida fuera del modelo?
- ¿La salida puede activar una acción equivocada?
- ¿Qué casos deben pasar a revisión humana?
- ¿El prompt contiene datos sensibles que no debería contener?

## Síntesis de la clase

- Un prompt avanzado es un contrato de tarea, contexto, restricciones y formato.
- Few-shot enseña un criterio con ejemplos, pero no garantiza exactitud.
- Una justificación breve y observable es más útil para una aplicación que pedir una cadena de pensamiento extensa.
- JSON válido no significa que los datos sean verdaderos ni suficientes.
- Un modelo System One (Jev) devuelve decisiones tipadas con probabilidades, sin texto libre: una alternativa directa al paso "prompt + parse".
- La incertidumbre (probabilidades y confianza) ayuda a decidir cuándo escalar a una persona.
- La validación local y la revisión humana siguen siendo parte del sistema, con cualquier modelo.

## Comprobación conceptual

1. ¿Qué diferencia hay entre zero-shot y few-shot?
2. ¿Por qué no alcanza con que `json.loads` no produzca un error?
3. ¿Qué señales deberían activar revisión humana?
4. ¿Qué diferencia hay entre un LLM con system prompt y un modelo System One como Jev?
5. ¿Qué información extra da Jev (probabilidades/confianza) y cómo la usarías?
6. ¿Qué parte del problema resuelve el prompting y cuál quedará para RAG?

## Puente con la Clase 6

Hoy organizamos una tarea y controlamos su salida, y vimos dos formas de obtener una decisión: un LLM que escribe y un modelo System One que decide. En la próxima clase agregaremos conocimiento externo: documentos, embeddings, búsqueda vectorial y reranking. Prompting define cómo usar el contexto; RAG decide qué contexto recuperar.